In [ ]:
Author: Tatum Thomas 

In [8]:
import pandas as pd
import numpy as np

# Read the main data
df_main = pd.read_csv("combined_calls_CAR.csv")
df_main['Activity Start Timestamp'] = pd.to_datetime(df_main['Activity Start Timestamp'])
df_main = df_main.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

print("Creating comprehensive front desk dataset...")

# ============================================================
# 1. IDENTIFY FRONT DESK SESSIONS AND ADD FLAGS
# ============================================================

# Mark front desk activities
df_main['is_front_desk'] = df_main['Activity Name'].str.contains('FrontDesk', case=False, na=False)

# Get sessions with front desk interactions
fd_sessions_list = df_main[df_main['is_front_desk']]['Contact Session ID'].unique()

# Add session-level flags
df_main['has_front_desk'] = df_main['Contact Session ID'].isin(fd_sessions_list)

# ============================================================
# 2. GET FRONT DESK TYPE FOR EACH SESSION
# ============================================================

# Get the front desk type for each session (first occurrence)
fd_types = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['Activity Name'].first()

# Merge back to main dataframe
df_main = df_main.merge(
    fd_types.rename('session_fd_type').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# ============================================================
# 3. ADD TIME COMPONENTS FOR FRONT DESK SESSIONS
# ============================================================

# Get first front desk timestamp for each session
fd_timestamps = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['Activity Start Timestamp'].first()

df_main = df_main.merge(
    fd_timestamps.rename('fd_timestamp').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# Extract time components from FD timestamp
df_main['fd_hour'] = df_main['fd_timestamp'].dt.hour
df_main['fd_day_of_week'] = df_main['fd_timestamp'].dt.day_name()
df_main['fd_day_of_week_num'] = df_main['fd_timestamp'].dt.dayofweek
df_main['fd_month'] = df_main['fd_timestamp'].dt.month_name()
df_main['fd_month_num'] = df_main['fd_timestamp'].dt.month
df_main['fd_year'] = df_main['fd_timestamp'].dt.year
df_main['fd_year_month'] = df_main['fd_timestamp'].dt.to_period('M').astype(str)  # Creates "2024-03" format
df_main['fd_month_year_display'] = df_main['fd_timestamp'].dt.strftime('%B %Y')  # Creates "March 2024" format

# ============================================================
# 4. CALCULATE FRONT DESK DURATION FOR EACH SESSION
# ============================================================

# Calculate session-level metrics
session_metrics = []

for session_id in fd_sessions_list:
    session_df = df_main[df_main['Contact Session ID'] == session_id].sort_values('Activity Start Timestamp')
    
    # Get FD timestamp and last timestamp
    fd_timestamp = session_df[session_df['is_front_desk']].iloc[0]['Activity Start Timestamp']
    last_timestamp = session_df.iloc[-1]['Activity Start Timestamp']
    
    # Calculate duration
    duration_seconds = (last_timestamp - fd_timestamp).total_seconds()
    duration_minutes = duration_seconds / 60
    
    session_metrics.append({
        'Contact Session ID': session_id,
        'fd_duration_seconds': duration_seconds,
        'fd_duration_minutes': duration_minutes,
        'session_start': session_df.iloc[0]['Activity Start Timestamp'],
        'session_end': last_timestamp,
        'total_activities': len(session_df)
    })

df_metrics = pd.DataFrame(session_metrics)

# Merge duration back to main dataframe
df_main = df_main.merge(df_metrics, on='Contact Session ID', how='left')

# ============================================================
# 5. ADD RETURN TO MAIN MENU ANALYSIS
# ============================================================

# Mark main menu EPs
main_menu_ep_names = [
    'Main Number Telephony EP',
    'Farmworker Main Number Telephony EP'
]
df_main['is_main_menu'] = df_main['EP Name'].isin(main_menu_ep_names)

# Create activity sequence
df_main['activity_sequence'] = df_main.groupby('Contact Session ID').cumcount()

# Find first FD position in each session
fd_positions = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['activity_sequence'].first()
df_main = df_main.merge(
    fd_positions.rename('first_fd_position').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# Mark activities after FD
df_main['after_fd'] = (
    (df_main['first_fd_position'].notna()) & 
    (df_main['activity_sequence'] > df_main['first_fd_position'])
)

# Find sessions that returned to main menu after FD
main_menu_returns = df_main[
    (df_main['after_fd'] == True) & 
    (df_main['is_main_menu'] == True)
]['Contact Session ID'].unique()

df_main['session_returned_to_main'] = df_main['Contact Session ID'].isin(main_menu_returns)

# ============================================================
# 6. ADD SUMMARY STATISTICS
# ============================================================

# Calculate summary stats by FD type
duration_stats = df_metrics.merge(
    df_main[['Contact Session ID', 'session_fd_type']].drop_duplicates(),
    on='Contact Session ID'
).groupby('session_fd_type')['fd_duration_minutes'].agg([
    ('avg_duration', 'mean'),
    ('median_duration', 'median'),
    ('min_duration', 'min'),
    ('max_duration', 'max')
]).round(2).reset_index()

# ============================================================
# 6.1. CALCULATE STAFF TIME ANALYSIS (CORRECTED)
# ============================================================

# FIXED: Get session counts by FD type (count unique sessions, not rows)
session_counts = df_main[df_main['has_front_desk']].groupby('session_fd_type')['Contact Session ID'].nunique().reset_index(name='session_count')

# Merge with duration stats
duration_stats = duration_stats.merge(session_counts, on='session_fd_type', how='left')

# Calculate total time per category (sessions * average duration)
duration_stats['total_time_minutes'] = duration_stats['session_count'] * duration_stats['avg_duration']

# Calculate overall totals
total_front_desk_time = duration_stats['total_time_minutes'].sum()
daily_front_desk_time = total_front_desk_time / 370

# Add these as columns to the main dataframe for Power BI
df_main['total_front_desk_time_all'] = total_front_desk_time
df_main['daily_front_desk_time_all'] = daily_front_desk_time

print(f"\nSTAFF TIME ANALYSIS (CORRECTED):")
print(f"Total front desk time: {total_front_desk_time:,.1f} minutes ({total_front_desk_time/60:,.1f} hours)")
print(f"Daily front desk time (370 weekdays): {daily_front_desk_time:.1f} minutes/day ({daily_front_desk_time/60:.1f} hours/day)")

# Debug output to verify correct session counts
print(f"\nSession count verification:")
for idx, row in duration_stats.iterrows():
    print(f"{row['session_fd_type']}: {row['session_count']} sessions × {row['avg_duration']:.1f} min = {row['total_time_minutes']:.1f} minutes")

# Merge stats back to main dataframe (CORRECTED)
df_main = df_main.merge(
    duration_stats[['session_fd_type', 'avg_duration', 'median_duration', 'min_duration', 'max_duration']],
    left_on='session_fd_type',
    right_on='session_fd_type',
    how='left'
)

# ============================================================
# 7. CLEAN UP AND EXPORT
# ============================================================

# Fill NaN values for non-FD sessions (UPDATED with new columns)
fill_columns = [
    'session_fd_type', 'fd_timestamp', 'fd_hour', 'fd_day_of_week', 
    'fd_day_of_week_num', 'fd_month', 'fd_month_num', 'fd_year', 'fd_year_month', 'fd_month_year_display',
    'fd_duration_seconds', 'fd_duration_minutes', 'session_start', 'session_end', 'total_activities',
    'first_fd_position', 'avg_duration', 'median_duration', 'min_duration', 'max_duration',
    'total_front_desk_time_all', 'daily_front_desk_time_all'
]

for col in fill_columns:
    if col in df_main.columns:
        if 'duration' in col or col in ['fd_hour', 'fd_day_of_week_num', 'fd_month_num', 'fd_year', 'total_activities', 'first_fd_position', 'total_front_desk_time_all', 'daily_front_desk_time_all']:
            df_main[col] = df_main[col].fillna(0)
        else:
            df_main[col] = df_main[col].fillna('N/A')

# Fill boolean columns
df_main['has_front_desk'] = df_main['has_front_desk'].fillna(False)
df_main['is_front_desk'] = df_main['is_front_desk'].fillna(False)
df_main['is_main_menu'] = df_main['is_main_menu'].fillna(False)
df_main['after_fd'] = df_main['after_fd'].fillna(False)
df_main['session_returned_to_main'] = df_main['session_returned_to_main'].fillna(False)

# Export the comprehensive dataset
df_main.to_csv('comprehensive_front_desk_data.csv', index=False)

print("=" * 60)
print("COMPREHENSIVE DATASET CREATED")
print("=" * 60)
print(f"Total rows: {len(df_main):,}")
print(f"Front desk sessions: {len(fd_sessions_list):,}")
print(f"Sessions returning to main: {len(main_menu_returns):,}")

print("\nNew columns added:")
print("   - has_front_desk (boolean)")
print("   - session_fd_type (FrontDeskTransfer, etc.)")
print("   - fd_timestamp (when FD transfer occurred)")
print("   - fd_hour, fd_day_of_week, fd_month, fd_month_year_display (time components)")
print("   - fd_duration_minutes (time spent on FD call)")
print("   - session_returned_to_main (boolean)")
print("   - avg_duration, median_duration (stats by FD type)")
print("   - total_front_desk_time_all, daily_front_desk_time_all (staff time metrics)")

print(f"\nExported: comprehensive_front_desk_data.csv ({len(df_main):,} rows)")

/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_2176/1797333762.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("combined_calls_CAR.csv")


Creating comprehensive front desk dataset...

STAFF TIME ANALYSIS (CORRECTED):
Total front desk time: 65,785.9 minutes (1,096.4 hours)
Daily front desk time (370 weekdays): 177.8 minutes/day (3.0 hours/day)

Session count verification:
FrontDeskTransfer: 2545 sessions × 5.0 min = 12750.4 minutes
FrontDeskTransfer1: 7912 sessions × 3.0 min = 23656.9 minutes
FrontDeskTransfer2: 6895 sessions × 4.1 min = 28269.5 minutes
FrontDeskTransfer3: 541 sessions × 2.0 min = 1109.0 minutes
COMPREHENSIVE DATASET CREATED
Total rows: 3,328,626
Front desk sessions: 17,893
Sessions returning to main: 15,146

New columns added:
   - has_front_desk (boolean)
   - session_fd_type (FrontDeskTransfer, etc.)
   - fd_timestamp (when FD transfer occurred)
   - fd_hour, fd_day_of_week, fd_month, fd_month_year_display (time components)
   - fd_duration_minutes (time spent on FD call)
   - session_returned_to_main (boolean)
   - avg_duration, median_duration (stats by FD type)
   - total_front_desk_time_all, daily_